# BERTopic + RAPIDS on Nemotron OpenImages

This notebook teaches topic modeling from first principles, then uses **BERTopic** with RAPIDS-accelerated **cuML UMAP** and **cuML HDBSCAN** on the `openimages_4` subset of [`nvidia/Nemotron-Image-Training-v3`](https://huggingface.co/datasets/nvidia/Nemotron-Image-Training-v3).

The core idea:

1. Treat each assistant-generated image analysis as a text document for readable topic labels.
2. Build a text baseline with a sentence-transformer.
3. Build the required image-embedding path with **NVIDIA C-RADIOv4**.
4. Use accelerated UMAP to make either embedding space easier to cluster.
5. Use accelerated HDBSCAN to find dense semantic groups and outliers.
6. Use BERTopic's c-TF-IDF layer to turn clusters into readable topics.

The original media for `openimages_4` is separate from the Hugging Face JSON/Parquet rows. The text baseline runs from the dataset alone. The C-RADIOv4 image-embedding section requires local OpenImages media, arranged so each row's `train/data/*.jpg` path resolves under `MEDIA_ROOT`.

Primary references:

- [Nemotron Image Training v3 dataset card](https://huggingface.co/datasets/nvidia/Nemotron-Image-Training-v3)
- [openimages_4 README](https://huggingface.co/datasets/nvidia/Nemotron-Image-Training-v3/blob/main/openimages_4/README.md)
- [NVIDIA C-RADIOv4-SO400M on Hugging Face](https://huggingface.co/nvidia/C-RADIOv4-SO400M)
- [NVLabs RADIO repository](https://github.com/NVlabs/RADIO)
- [RAPIDS on Google Colab](https://docs.rapids.ai/deployment/stable/platforms/colab/)
- [BERTopic documentation](https://maartengr.github.io/BERTopic/)


## 0. Runtime expectations

Use **Runtime > Change runtime type > GPU** before running this notebook.

The default sample size is large enough to produce real topics but still intended for a Colab GPU:

- `N_DOCS = 50_000` for the main run.
- `N_IMAGE_DOCS = 512` for the C-RADIOv4 image-embedding run.
- `LOW_MEMORY_N_DOCS = 10_000` if Colab memory is tight.
- Full `openimages_4` has about 504K rows, and the referenced media is not bundled with the text rows.

If a RAPIDS install cell asks you to restart the runtime, restart and rerun from the imports/checks section. If the C-RADIOv4 section reports no local images, attach or mount OpenImages media under `/content/openimages_media` before running the image-embedding cells.


## 1. First principles

Topic modeling asks: **what recurring themes exist in a pile of documents?**

BERTopic is a pipeline of four ideas:

1. **Embeddings:** convert documents into vectors where similar meanings are near each other.
2. **UMAP:** compress those high-dimensional vectors while preserving useful local neighborhoods.
3. **HDBSCAN:** find dense groups of documents and mark ambiguous documents as outliers.
4. **c-TF-IDF:** summarize each discovered cluster with words and phrases that distinguish it from the others.

The toy corpus below is intentionally tiny. It gives us a mental model before we use GPU acceleration and a large multimodal training dataset.


In [1]:
toy_documents = [
    "A crowded parade moves down a city street with flags and spectators.",
    "People gather at a festival while performers march past the crowd.",
    "A sandwich and coffee sit on a wooden cafe table.",
    "A bowl of noodles with chopsticks is ready to eat.",
    "A bronze statue stands in a museum courtyard.",
    "An artist paints a figure inside a quiet studio.",
]

for i, doc in enumerate(toy_documents, start=1):
    print(f"{i}. {doc}")


1. A crowded parade moves down a city street with flags and spectators.
2. People gather at a festival while performers march past the crowd.
3. A sandwich and coffee sit on a wooden cafe table.
4. A bowl of noodles with chopsticks is ready to eat.
5. A bronze statue stands in a museum courtyard.
6. An artist paints a figure inside a quiet studio.


In the full BERTopic pipeline, HDBSCAN will discover clusters automatically. For this first-principles sketch, we manually assign toy groups so we can inspect the final c-TF-IDF idea without installing anything yet.


In [2]:
from collections import Counter, defaultdict
import math
import re

toy_topic_labels = {
    0: "public events",
    1: "food",
    2: "art objects",
}

toy_assignments = [0, 0, 1, 1, 2, 2]


def tokenize(text):
    return re.findall(r"[a-z]{3,}", text.lower())


def simple_class_tfidf(documents, assignments, top_n=5):
    grouped = defaultdict(list)
    for document, topic_id in zip(documents, assignments):
        grouped[topic_id].append(document)

    topic_word_counts = {
        topic_id: Counter(token for doc in docs for token in tokenize(doc))
        for topic_id, docs in grouped.items()
    }
    all_topics = list(topic_word_counts)
    words = sorted({word for counts in topic_word_counts.values() for word in counts})

    rows = []
    for topic_id, counts in topic_word_counts.items():
        total = sum(counts.values())
        scores = []
        for word in words:
            tf = counts[word] / total if total else 0
            topics_with_word = sum(word in topic_word_counts[other] for other in all_topics)
            idf = math.log((1 + len(all_topics)) / (1 + topics_with_word)) + 1
            scores.append((word, tf * idf))
        rows.append(
            (
                topic_id,
                toy_topic_labels[topic_id],
                [word for word, score in sorted(scores, key=lambda item: item[1], reverse=True)[:top_n]],
            )
        )
    return rows


for topic_id, label, words in simple_class_tfidf(toy_documents, toy_assignments):
    print(f"Topic {topic_id} ({label}): {', '.join(words)}")


Topic 0 (public events): city, crowd, crowded, down, festival
Topic 1 (food): bowl, cafe, chopsticks, coffee, eat
Topic 2 (art objects): artist, bronze, courtyard, figure, inside


The simplified c-TF-IDF output is not the same as a full BERTopic model, but it reveals the teaching intuition: **clusters become interpretable only after we ask which terms distinguish one cluster from all the other clusters.**


## 2. Colab GPU setup

RAPIDS provides GPU implementations of tools that are commonly CPU-bound in topic modeling workflows. In this notebook, the important pieces are:

- `cuml.manifold.UMAP`
- `cuml.cluster.HDBSCAN`

The RAPIDS Colab docs recommend using their pip installer utility. This cell can take several minutes.


In [3]:
!nvidia-smi


Tue Jul 14 15:14:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# RAPIDS install, following the official RAPIDS Colab guidance:
# https://docs.rapids.ai/deployment/stable/platforms/colab/
!if [ ! -d rapidsai-csp-utils ]; then git clone https://github.com/rapidsai/rapidsai-csp-utils.git; fi
!python rapidsai-csp-utils/colab/pip-install.py

# Notebook-level dependencies.
!pip install -q bertopic sentence-transformers datasets pandas plotly pillow pyarrow scikit-learn transformers torchvision


Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 700, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 700 (delta 186), reused 139 (delta 139), pack-reused 484 (from 1)
Receiving objects: 100% (700/700), 234.66 KiB | 8.09 MiB/s, done.
Resolving deltas: 100% (370/370), done.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.12.13 environment at: /usr
Resolved 180 packages in 1.99s
Prepared 10 packages in 1.28s
Uninstalled 4 packages in 238ms
Installed 10 packages in 51ms
 - bokeh==3.8.2
 + bokeh==3.6.3
 + cugraph-cu12==26.2.0
 + cuxfilter-cu12==26.2.0
 + datashader==0.19.1
 - holoviews==1.23.0
 + holoviews==1.20.2
 + jupyter-server-proxy==4.5.0
 - panel==1.9.3
 + panel==1.7.5
 + pyct==0.6.0
 - shapely==2.1.2
 + shapely==2.0.7
 + simpervisor==1.0.0

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do no

In [5]:
!pip install datasets==5.0.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [6]:
import os
import random
import re
import subprocess
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image as PILImage

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoModel, CLIPImageProcessor

from cuml.manifold import UMAP as cuUMAP
from cuml.cluster import HDBSCAN as cuHDBSCAN

try:
    import cupy as cp
except ImportError:
    cp = None

print("Core imports succeeded.")


Core imports succeeded.


In [7]:
def assert_gpu_available():
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    assert result.returncode == 0, "No NVIDIA GPU detected. In Colab, choose Runtime > Change runtime type > GPU."
    print(result.stdout.splitlines()[0])


assert_gpu_available()


Tue Jul 14 15:15:52 2026       


## 3. Load Nemotron OpenImages

The dataset is a collection of multimodal conversation rows. For `openimages_4`, each row has:

- `id`: a UUID-like row identifier.
- `messages`: a list of chat messages.

Inside `messages`, the user message includes an image reference such as `train/data/7cc57a1f9569a842.jpg`, and the assistant message contains a long visual description or analysis. We will cluster those assistant descriptions.


In [8]:
DATASET_NAME = "nvidia/Nemotron-Image-Training-v3"
DATASET_CONFIG = "openimages_4"
SPLIT = "train"
IMAGE_EMBEDDING_MODEL = "nvidia/C-RADIOv4-SO400M"

N_DOCS = 50_000
N_IMAGE_DOCS = 512
LOW_MEMORY_N_DOCS = 10_000
RANDOM_SEED = 42

UMAP_N_NEIGHBORS = 15
UMAP_N_COMPONENTS = 5
UMAP_MIN_DIST = 0.0
MIN_CLUSTER_SIZE = 60
MIN_SAMPLES = 10
VECTORIZER_MIN_DF = 5
IMAGE_MIN_CLUSTER_SIZE = 12
IMAGE_VECTORIZER_MIN_DF = 2
IMAGE_BATCH_SIZE = 4

MEDIA_ROOT = Path("/content/openimages_media")


In [9]:
def load_openimages4(split=SPLIT, streaming=True):
    '''Load the Nemotron OpenImages v4 subset.

    Streaming keeps Colab from eagerly materializing all 504K rows. For teaching and
    sampling, this is usually the most forgiving default.
    '''
    return load_dataset(
        DATASET_NAME,
        DATASET_CONFIG,
        split=split,
        streaming=streaming,
    )


preview_ds = load_openimages4()
preview_rows = list(preview_ds.take(3))

print(f"Previewed {len(preview_rows)} rows from {DATASET_NAME}/{DATASET_CONFIG}/{SPLIT}.")
print(preview_rows[0].keys())
preview_rows[0]["messages"]


README.md:   0%|          | 0.00/129k [00:00<?, ?B/s]

Previewed 3 rows from nvidia/Nemotron-Image-Training-v3/openimages_4/train.
dict_keys(['id', 'messages'])


[{'role': 'user',
  'content': [{'type': 'image', 'image': 'train/data/7cc57a1f9569a842.jpg'},
   '\nWhat do you think is going on in this snapshot?']},
 {'role': 'assistant',
  'content': ["Based on the visual cues in the image, here is a detailed analysis of what is likely happening:\n\nThis snapshot captures a candid moment of a group of people, likely tourists or students, on an outdoor excursion. The scene is set on a bridge or a riverside walkway, with a calm river and lush green trees in the background.\n\nHere's a breakdown of the key elements and what they suggest:\n\n-   **The Central Figure:** A young woman with long dark hair, wearing a black t-shirt, black pants, and a striped tote bag, is the focal point. She is standing, leaning against a metal railing, and actively speaking. Her hands are gesturing, which is a strong indicator that she is explaining something. Her facial expression is engaged and focused on the people around her. This suggests she is in the role of a gu

In [10]:
def assert_dataset_shape(rows):
    assert rows, "Expected at least one preview row."
    first = rows[0]
    assert "id" in first, "Dataset row is missing 'id'."
    assert "messages" in first, "Dataset row is missing 'messages'."
    assert isinstance(first["messages"], list), "'messages' should be a list."
    assert len(first["messages"]) >= 2, "Expected at least user and assistant messages."
    print("Dataset shape check passed.")


assert_dataset_shape(preview_rows)


Dataset shape check passed.


## 4. Turn multimodal conversations into documents

BERTopic expects documents. The Nemotron rows are conversations, so we need a careful extraction step.

For each row we keep:

- `image_path`: the referenced media path, for optional visual inspection later.
- `user_prompt`: the user instruction or question.
- `assistant_description`: the generated image description/analysis.
- `document_text`: the exact text we will cluster; here it defaults to `assistant_description`.


In [11]:
def _iter_content_parts(content):
    if isinstance(content, list):
        for item in content:
            yield item
    else:
        yield content


def _extract_text(content):
    text_parts = []
    for part in _iter_content_parts(content):
        if isinstance(part, str):
            text_parts.append(part)
    return "\n".join(text_parts).strip()


def _extract_image_path(content):
    for part in _iter_content_parts(content):
        if isinstance(part, dict) and part.get("type") == "image":
            image = part.get("image")
            if image:
                return image
    return None


def extract_message_fields(row):
    '''Convert one Nemotron multimodal conversation row into a topic-modeling record.'''
    messages = row.get("messages", [])
    user_message = next((message for message in messages if message.get("role") == "user"), {})
    assistant_message = next((message for message in messages if message.get("role") == "assistant"), {})

    user_content = user_message.get("content", [])
    assistant_content = assistant_message.get("content", [])

    image_path = _extract_image_path(user_content)
    user_prompt = _extract_text(user_content)
    assistant_description = _extract_text(assistant_content)
    document_text = assistant_description.strip()

    return {
        "id": row.get("id"),
        "image_path": image_path,
        "user_prompt": user_prompt,
        "assistant_description": assistant_description,
        "document_text": document_text,
    }


for record in [extract_message_fields(row) for row in preview_rows[:2]]:
    print("id:", record["id"])
    print("image_path:", record["image_path"])
    print("prompt:", record["user_prompt"][:120].replace("\n", " "))
    print("description:", record["assistant_description"][:240].replace("\n", " "))
    print("-" * 80)


id: f3e4ed27-d0c9-480c-bcb2-9ab66cf1bb77
image_path: train/data/7cc57a1f9569a842.jpg
prompt: What do you think is going on in this snapshot?
description: Based on the visual cues in the image, here is a detailed analysis of what is likely happening:  This snapshot captures a candid moment of a group of people, likely tourists or students, on an outdoor excursion. The scene is set on a bridge
--------------------------------------------------------------------------------
id: 08e2b621-61f4-4a01-8f30-3154cf22784f
image_path: train/data/517f6cbf74e03101.jpg
prompt: What do you think is going on in this snapshot?
description: Based on the image provided, here is an analysis of what is happening in the snapshot.  The image captures a close-up photograph of a single, multi-sided die resting on a dark, textured surface. The die is the central focus of the image.  #
--------------------------------------------------------------------------------


In [12]:
from IPython.display import Image, display
from pathlib import Path

specific_image_path = Path("/content/openimages_media/train/data/7cc57a1f9569a842.jpg")

if specific_image_path.exists():
    display(Image(filename=str(specific_image_path)))
else:
    print(f"Image not found at {specific_image_path}.")
    print("Please ensure the openimages_4 media is mounted or downloaded to this path.")

Image not found at /content/openimages_media/train/data/7cc57a1f9569a842.jpg.
Please ensure the openimages_4 media is mounted or downloaded to this path.


In [13]:
def sample_documents(dataset, n_docs=N_DOCS, seed=RANDOM_SEED, buffer_size=10_000):
    '''Sample rows reproducibly enough for an educational streaming workflow.'''
    shuffled = dataset.shuffle(seed=seed, buffer_size=buffer_size)
    records = []
    for row in shuffled.take(n_docs):
        record = extract_message_fields(row)
        if record["document_text"]:
            records.append(record)
    return pd.DataFrame(records)


ds = load_openimages4()
df = sample_documents(ds, n_docs=N_DOCS, seed=RANDOM_SEED)

print(df.shape)
df.head(3)


(50000, 5)


,id,image_path,user_prompt,assistant_description,document_text
0,9687a4d5-d844-4164-92e7-2e4597b35458,train/data/f98a730ba8d8194c.jpg,Write a detailed description of the given image.,This is a wide-angle photograph capturing a sc...,This is a wide-angle photograph capturing a sc...
1,1b9fdac8-2381-4884-a1cf-f6da50e4a171,train/data/3ac0c1a4925e0d4c.jpg,What do you see happening in this image?,"Based on the image provided, here is a descrip...","Based on the image provided, here is a descrip..."
2,741c02d1-e607-43ea-aede-aac0884012a8,train/data/5c644c8392a4f144.jpg,Analyze the image in a comprehensive and detai...,This is a detailed analysis of the provided im...,This is a detailed analysis of the provided im...


In [14]:
def assert_extracted_text_quality(df, min_rows=100, min_average_chars=500):
    assert len(df) >= min_rows, f"Expected at least {min_rows} extracted rows, got {len(df)}."
    assert df["document_text"].notna().all(), "document_text contains missing values."
    assert (df["document_text"].str.len() > 0).all(), "document_text contains empty strings."
    average_chars = df["document_text"].str.len().mean()
    assert average_chars >= min_average_chars, (
        f"Average document length looks too short: {average_chars:.1f} characters."
    )
    print(f"Text extraction check passed. Average chars/document: {average_chars:,.1f}")


assert_extracted_text_quality(df)

docs = df["document_text"].tolist()


Text extraction check passed. Average chars/document: 2,500.5


## 5. Accelerated BERTopic

Now we build the real pipeline:

1. `SentenceTransformer` creates semantic vectors from descriptions.
2. `cuUMAP` reduces those vectors on the GPU.
3. `cuHDBSCAN` clusters those reduced vectors on the GPU.
4. `BERTopic` uses those cluster labels to compute readable topic descriptions.




In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    docs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embeddings = embeddings.astype("float32")
print(embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [ ]:
def assert_embeddings_match_documents(embeddings, docs):
    assert len(embeddings) == len(docs), (
        f"Embedding/document mismatch: {len(embeddings)} embeddings for {len(docs)} documents."
    )
    assert embeddings.ndim == 2, "Expected a 2D embedding matrix."
    assert np.isfinite(embeddings).all(), "Embeddings contain non-finite values."
    print("Embedding check passed.")


assert_embeddings_match_documents(embeddings, docs)


In [ ]:
def build_gpu_topic_model(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=UMAP_N_COMPONENTS,
    min_dist=UMAP_MIN_DIST,
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    vectorizer_min_df=VECTORIZER_MIN_DF,
    random_state=RANDOM_SEED,
):
    umap_model = cuUMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric="cosine",
        random_state=random_state,
        verbose=True,
    )

    hdbscan_model = cuHDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        verbose=True,
    )

    vectorizer_model = CountVectorizer(
        stop_words="english",
        min_df=vectorizer_min_df,
        ngram_range=(1, 2),
    )

    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        embedding_model=None,
        calculate_probabilities=False,
        verbose=True,
    )


topic_model = build_gpu_topic_model()
topics, probabilities = topic_model.fit_transform(docs, embeddings)
print(f"Modeled {len(docs):,} documents.")


In [ ]:
def assert_topic_model_has_topics(topics):
    topic_ids = set(topics)
    non_outlier_topics = sorted(topic_id for topic_id in topic_ids if topic_id != -1)
    assert len(non_outlier_topics) > 1, (
        f"Expected more than one non-outlier topic, got {len(non_outlier_topics)}."
    )
    outlier_share = np.mean(np.array(topics) == -1)
    print(f"Topic check passed. Non-outlier topics: {len(non_outlier_topics):,}; outlier share: {outlier_share:.1%}")


assert_topic_model_has_topics(topics)


## 6. Interpret topics

Topic modeling is only useful if we inspect it skeptically. We will look at:

- frequent topics,
- top c-TF-IDF terms,
- representative documents,
- outlier share,
- a 2D projection for visual scanning.


In [ ]:
def summarize_topics(topic_model, docs, topics, top_n=20):
    topic_info = topic_model.get_topic_info()
    outlier_share = float(np.mean(np.array(topics) == -1))
    visible = topic_info.head(top_n).copy()
    visible["outlier_share"] = outlier_share
    return visible


topic_summary = summarize_topics(topic_model, docs, topics)
topic_summary


In [ ]:
for topic_id in topic_summary.loc[topic_summary["Topic"] != -1, "Topic"].head(5):
    print(f"\nTopic {topic_id}")
    print(topic_model.get_topic(topic_id)[:10])


In [ ]:
def show_representative_examples(topic_model, df, topic_id, n=3, media_root=MEDIA_ROOT):
    representative_docs = topic_model.get_representative_docs(topic_id)[:n]
    for rank, doc in enumerate(representative_docs, start=1):
        matches = df[df["document_text"] == doc]
        row = matches.iloc[0] if len(matches) else None

        print(f"\nTopic {topic_id} example {rank}")
        if row is not None:
            print("row id:", row["id"])
            print("image path:", row["image_path"])
            candidate = media_root / str(row["image_path"]) if row["image_path"] else None
            if candidate and candidate.exists():
                from IPython.display import display, Image

                display(Image(filename=str(candidate), width=360))
            elif row is not None and row["image_path"]:
                print(f"(Image not found locally under {media_root}; skipping display.)")

        print(textwrap.shorten(doc.replace("\n", " "), width=900, placeholder=" ..."))


first_real_topic = int(topic_summary.loc[topic_summary["Topic"] != -1, "Topic"].iloc[0])
show_representative_examples(topic_model, df, first_real_topic, n=3)


In [ ]:
# Build a separate 2D GPU UMAP projection for interactive visualization.
umap_2d = cuUMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=2,
    min_dist=0.05,
    metric="cosine",
    random_state=RANDOM_SEED,
)

embeddings_2d = umap_2d.fit_transform(embeddings)
if hasattr(embeddings_2d, "get"):
    embeddings_2d = embeddings_2d.get()
elif cp is not None and isinstance(embeddings_2d, cp.ndarray):
    embeddings_2d = cp.asnumpy(embeddings_2d)

fig = topic_model.visualize_documents(
    docs,
    topics=topics,
    reduced_embeddings=np.asarray(embeddings_2d),
    hide_document_hover=True,
    custom_labels=True,
)
fig


## 7. C-RADIOv4 image embeddings

Now we switch from language-derived vectors to image-derived vectors. C-RADIOv4 is a vision backbone from NVIDIA Research. The default checkpoint here is `nvidia/C-RADIOv4-SO400M`, the smaller v4 variant, because it is more realistic for Colab than the larger `nvidia/C-RADIOv4-H` checkpoint.

Important distinction:

- The **documents** still come from assistant descriptions, because BERTopic needs text to create topic words.
- The **embeddings** in this section come from the actual image pixels, using C-RADIOv4.

This section requires local image files. The dataset rows reference paths such as:

```text
train/data/7cc57a1f9569a842.jpg
```

Place OpenImages media under:

```text
/content/openimages_media/train/data/*.jpg
```

The notebook will fail fast here if no sampled image files are available, because the requirement is to create image embeddings with C-RADIOv4 rather than silently fall back to text embeddings.


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

OPENIMAGES_MIRROR_BASE = "https://open-images-dataset.s3.amazonaws.com"

# Get the first 1000 non-null image paths from our sampled dataframe
images_to_download = df["image_path"].dropna().head(1000).tolist()

def openimages_source_url(img_path):
    path = Path(str(img_path))
    if len(path.parts) < 2:
        raise ValueError(f"Unexpected OpenImages media path: {img_path!r}")
    split = path.parts[0]
    return f"{OPENIMAGES_MIRROR_BASE}/{split}/{path.name}"


def download_image(img_path):
    dest_path = MEDIA_ROOT / str(img_path)
    source_url = openimages_source_url(img_path)

    if dest_path.exists():
        return {"image_path": str(img_path), "status": "skipped", "url": source_url}

    dest_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = dest_path.with_name(dest_path.name + ".part")

    try:
        with requests.get(source_url, stream=True, timeout=(10, 60)) as response:
            response.raise_for_status()
            with temp_path.open("wb") as output_file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

        with PILImage.open(temp_path) as image:
            image.verify()

        temp_path.replace(dest_path)
        return {"image_path": str(img_path), "status": "downloaded", "url": source_url}
    except Exception as exc:
        temp_path.unlink(missing_ok=True)
        return {
            "image_path": str(img_path),
            "status": "error",
            "url": source_url,
            "error": f"{type(exc).__name__}: {exc}",
        }

print(f"Attempting to download {len(images_to_download)} images to {MEDIA_ROOT}...")

with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(tqdm(executor.map(download_image, images_to_download), total=len(images_to_download)))

successful_downloads = sum(result["status"] == "downloaded" for result in results)
skipped_downloads = sum(result["status"] == "skipped" for result in results)
failed_downloads = [result for result in results if result["status"] == "error"]

print(f"\nDownloaded {successful_downloads} images; skipped {skipped_downloads} existing files; {len(failed_downloads)} failed.")
for failure in failed_downloads[:10]:
    print(f"- {failure['image_path']}: {failure['error']} ({failure['url']})")
if len(failed_downloads) > 10:
    print(f"... and {len(failed_downloads) - 10} more failures.")
if successful_downloads + skipped_downloads == 0:
    print("No images are available locally. Review the per-file errors above.")

In [ ]:
def resolve_local_image_path(image_path, media_root=MEDIA_ROOT):
    if not image_path:
        return None
    candidate = media_root / str(image_path)
    return candidate if candidate.exists() else None


def count_available_images(df, media_root=MEDIA_ROOT, n=1000):
    checked = df["image_path"].dropna().head(n)
    existing = sum(resolve_local_image_path(path, media_root) is not None for path in checked)
    return existing, len(checked)


def available_image_records(df, media_root=MEDIA_ROOT, max_images=N_IMAGE_DOCS):
    image_df = df.copy()
    image_df["local_image_path"] = image_df["image_path"].apply(
        lambda path: resolve_local_image_path(path, media_root)
    )
    image_df = image_df[image_df["local_image_path"].notna()].head(max_images).copy()
    image_df["local_image_path"] = image_df["local_image_path"].astype(str)
    return image_df.reset_index(drop=True)


existing, checked = count_available_images(df)
print(f"Found {existing} local images among the first {checked} sampled image references under {MEDIA_ROOT}.")

image_df = available_image_records(df)
if image_df.empty:
    raise FileNotFoundError(
        "C-RADIOv4 image embeddings require local image files. "
        f"Mount or copy OpenImages media so paths like {MEDIA_ROOT}/train/data/*.jpg exist."
    )

image_docs = image_df["document_text"].tolist()
image_paths = image_df["local_image_path"].tolist()
image_df.head(3)


In [ ]:
!pip install open-clip-torch==3.3.0

In [ ]:
from tqdm.notebook import tqdm

def load_cradiov4_model(device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    image_processor = CLIPImageProcessor.from_pretrained(IMAGE_EMBEDDING_MODEL)
    image_model = AutoModel.from_pretrained(IMAGE_EMBEDDING_MODEL, trust_remote_code=True, torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32)
    image_model.eval().to(device)
    return image_processor, image_model, device


def cradio_preferred_processor_size(image_model):
    preferred_height, preferred_width = (
        int(value) for value in image_model.preferred_resolution
    )
    supported_height, supported_width = (
        int(value)
        for value in image_model.get_nearest_supported_resolution(
            preferred_height, preferred_width
        )
    )
    preferred_size = (preferred_height, preferred_width)
    supported_size = (supported_height, supported_width)
    if supported_size != preferred_size:
        raise ValueError(
            f"C-RADIO preferred resolution {preferred_size} is not supported; "
            f"nearest supported resolution is {supported_size}."
        )
    return {"height": preferred_height, "width": preferred_width}


def _cradiov4_summary_from_output(output):
    if isinstance(output, tuple):
        return output[0]
    if hasattr(output, "summary"):
        return output.summary
    if isinstance(output, dict):
        backbone = output.get("backbone", next(iter(output.values())))
        if isinstance(backbone, tuple):
            return backbone[0]
        if hasattr(backbone, "summary"):
            return backbone.summary
    raise TypeError(f"Could not extract a C-RADIOv4 summary embedding from output type {type(output)!r}.")


def embed_images_with_cradiov4(image_paths, image_processor, image_model, device, batch_size=IMAGE_BATCH_SIZE):
    all_embeddings = []
    model_dtype = next(image_model.parameters()).dtype
    processor_size = cradio_preferred_processor_size(image_model)
    target_size = (processor_size["height"], processor_size["width"])
    print(f"C-RADIO input resolution: {target_size[0]}x{target_size[1]}")

    for start in tqdm(range(0, len(image_paths), batch_size), desc="Embedding images"):
        batch_paths = image_paths[start : start + batch_size]
        images = []
        for path in batch_paths:
            with PILImage.open(path) as image:
                images.append(image.convert("RGB"))

        pixel_values = image_processor(
            images=images,
            return_tensors="pt",
            do_resize=True,
            size=processor_size,
        ).pixel_values
        expected_batch_shape = (len(images), 3, *target_size)
        assert tuple(pixel_values.shape) == expected_batch_shape, (
            f"Unexpected processed batch shape {tuple(pixel_values.shape)}; "
            f"expected {expected_batch_shape}."
        )
        pixel_values = pixel_values.to(device=device, dtype=model_dtype)

        with torch.inference_mode():
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
                output = image_model(pixel_values)
                summary = _cradiov4_summary_from_output(output)
                summary = torch.nn.functional.normalize(summary.float(), dim=-1)

        all_embeddings.append(summary.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0).astype("float32")


cradio_processor, cradio_model, cradio_device = load_cradiov4_model()


In [ ]:
# Regenerate image embeddings to see the tqdm progress bar
image_embeddings = embed_images_with_cradiov4(
    image_paths,
    cradio_processor,
    cradio_model,
    cradio_device,
)
print(image_embeddings.shape)

In [ ]:
def assert_image_embeddings_match_images(image_embeddings, image_df):
    assert len(image_embeddings) == len(image_df), (
        f"Image embedding/image mismatch: {len(image_embeddings)} embeddings for {len(image_df)} images."
    )
    assert image_embeddings.ndim == 2, "Expected a 2D image embedding matrix."
    assert np.isfinite(image_embeddings).all(), "Image embeddings contain non-finite values."
    assert (np.linalg.norm(image_embeddings, axis=1) > 0).all(), "Image embeddings contain zero vectors."
    print("C-RADIOv4 image embedding check passed.")


assert_image_embeddings_match_images(image_embeddings, image_df)


## 8. Image-embedding BERTopic

This is the key multimodal move: BERTopic will use the **C-RADIOv4 image embeddings** to decide which samples belong together, then use the paired assistant descriptions to label those image-driven clusters.

That gives us topics such as visual settings, object categories, events, layouts, or photographic styles, depending on what C-RADIOv4 groups together.


In [ ]:
image_topic_model = build_gpu_topic_model(
    min_cluster_size=IMAGE_MIN_CLUSTER_SIZE,
    min_samples=max(3, IMAGE_MIN_CLUSTER_SIZE // 3),
    vectorizer_min_df=IMAGE_VECTORIZER_MIN_DF,
)

image_topics, image_probabilities = image_topic_model.fit_transform(image_docs, image_embeddings)
assert_topic_model_has_topics(image_topics)

image_topic_summary = summarize_topics(image_topic_model, image_docs, image_topics)
image_topic_summary


In [ ]:
first_image_topic = int(image_topic_summary.loc[image_topic_summary["Topic"] != -1, "Topic"].iloc[0])
show_representative_examples(image_topic_model, image_df, first_image_topic, n=3)


## 9. Optional image inspection

The representative examples above display images when the files exist under `MEDIA_ROOT`. This final helper is useful when you want to manually audit a topic by sampling more examples from a C-RADIOv4 image cluster.


In [ ]:
def show_topic_image_grid(image_topic_model, image_df, topic_id, n=9, media_root=MEDIA_ROOT):
    from IPython.display import display

    topic_docs = image_topic_model.get_representative_docs(topic_id)[:n]
    for doc in topic_docs:
        matches = image_df[image_df["document_text"] == doc]
        if not len(matches):
            continue
        row = matches.iloc[0]
        path = resolve_local_image_path(row["image_path"], media_root)
        if path is not None:
            print(row["image_path"])
            display(PILImage.open(path).convert("RGB").resize((256, 256)))


show_topic_image_grid(image_topic_model, image_df, first_image_topic, n=6)


## 10. Sanity checks

These checks summarize the assumptions the notebook relies on. If one fails, read the error as a diagnosis of the pipeline stage that needs attention.


In [ ]:
assert_gpu_available()
assert_dataset_shape(preview_rows)
assert_extracted_text_quality(df)
assert_embeddings_match_documents(embeddings, docs)
assert_topic_model_has_topics(topics)
assert_image_embeddings_match_images(image_embeddings, image_df)
assert_topic_model_has_topics(image_topics)

print("All notebook sanity checks passed.")


## Where to go next

Good follow-up experiments:

- Compare `document_text = assistant_description` against `document_text = user_prompt + assistant_description`.
- Lower `MIN_CLUSTER_SIZE` to surface more specific topics.
- Increase `N_DOCS` if you have a larger GPU.
- Increase `N_IMAGE_DOCS` after confirming the C-RADIOv4 path works on your GPU.
- Compare `nvidia/C-RADIOv4-SO400M` against `nvidia/C-RADIOv4-H` if you have enough VRAM.
- Audit whether C-RADIOv4 image clusters correspond to meaningful visual patterns.
- Compare cuML UMAP/HDBSCAN against CPU UMAP/HDBSCAN on a smaller sample to measure speed and topic stability.
